# Hyperspectral Image Classification with CNN and Transformer

**Author:** Alessandro Carotenuto  
**Student ID:** 1847282  
**Dataset:** Pavia University  

## Project aim

The project aims to classify land-cover pixels in a hyperspectral image.
A spatial-spectral patch centered on each labeled pixel is used as input.
A CNN learns local features, while a lightweight Transformer models non-local context.

## Setup 

### Paths

In [13]:
import numpy as np

project_root = Path.cwd().parent
data_dir = project_root / "data" / "raw"

cube_path = data_dir / "PaviaU.mat"
ground_truth_path = data_dir / "PaviaU_gt.mat"

print("Cube:", cube_path)
print("Cube exists:", cube_path.exists())

print("Ground truth:", ground_truth_path)
print("Ground truth exists:", ground_truth_path.exists())

Cube: c:\Users\alex1\Documents\Cloned Repositories\Hyperspectral-Neural-Networks-Project\data\raw\PaviaU.mat
Cube exists: True
Ground truth: c:\Users\alex1\Documents\Cloned Repositories\Hyperspectral-Neural-Networks-Project\data\raw\PaviaU_gt.mat
Ground truth exists: True


### Loading MATLAB Files

In [14]:
from scipy.io import loadmat

cube_file = loadmat(cube_path)
ground_truth_file = loadmat(ground_truth_path)

print(cube_file.keys())
print(ground_truth_file.keys())

dict_keys(['__header__', '__version__', '__globals__', 'paviaU'])
dict_keys(['__header__', '__version__', '__globals__', 'paviaU_gt'])


### Dimension Check

In [15]:
cube = cube_file["paviaU"]
ground_truth = ground_truth_file["paviaU_gt"]

print("Cube:", cube.shape, cube.dtype)
print("Ground truth:", ground_truth.shape, ground_truth.dtype)
print("Cube range:", cube.min(), cube.max())
print("Labels:", np.unique(ground_truth))

Cube: (610, 340, 103) uint16
Ground truth: (610, 340) uint8
Cube range: 0 8000
Labels: [0 1 2 3 4 5 6 7 8 9]


### Class Count

In [16]:
class_names = {
    0: "Unlabeled",
    1: "Asphalt",
    2: "Meadows",
    3: "Gravel",
    4: "Trees",
    5: "Painted metal sheets",
    6: "Bare Soil",
    7: "Bitumen",
    8: "Self-Blocking Bricks",
    9: "Shadows",
}

labels, counts = np.unique(ground_truth, return_counts=True)

for label, count in zip(labels, counts):
    print(label, class_names[label], count)

0 Unlabeled 164624
1 Asphalt 6631
2 Meadows 18649
3 Gravel 2099
4 Trees 3064
5 Painted metal sheets 1345
6 Bare Soil 5029
7 Bitumen 1330
8 Self-Blocking Bricks 3682
9 Shadows 947


### Enum to define PATCHSIZE

In [17]:
from enum import IntEnum


class PatchSize(IntEnum):
    SMALLEST = 7
    SMALL = 9
    MEDIUM = 11
    LARGE = 15
    LARGEST = 25


PATCH_SIZE = PatchSize.MEDIUM
PATCH_RADIUS = PATCH_SIZE // 2

print("Patch size:", int(PATCH_SIZE))
print("Patch radius:", PATCH_RADIUS)

Patch size: 11
Patch radius: 5


### Grid Varibales

In [19]:
height, width, bands = cube.shape

grid_rows = 10
grid_cols = 10

block_height = height // grid_rows
block_width = width // grid_cols

print(height, width, bands)
print(block_height, block_width)

610 340 103
61 34


### Block IDs

In [20]:
row_block = np.arange(height) // block_height
col_block = np.arange(width) // block_width

block_ids = (
    row_block[:, None] * grid_cols
    + col_block[None, :]
)

unique_blocks, pixels_per_block = np.unique(
    block_ids,
    return_counts=True,
)

print("Block map shape:", block_ids.shape)
print("Number of blocks:", len(unique_blocks))
print("Pixels per block:", pixels_per_block.min(), pixels_per_block.max())

Block map shape: (610, 340)
Number of blocks: 100
Pixels per block: 2074 2074
